In [3]:
import tdmclient.notebook
await tdmclient.notebook.start()

In [ ]:
%%run_python

# === Parameters (integers only) ===
SPEED = 200
TURN_SPEED = 200
STOP_DIST = 3200        # front sensor: ≈ 2 cm (adjust)
TURN_TICKS = 90         # 10 ms ticks for ≈ 90° (calibrate)
PASS_TICKS = 300        # ticks to drive past the edge before turning (calibrate)
NB_TURNS = 2            # stop after this many right turns at corners
FINISH_TICKS = 100      # ticks driven after the last turn before stopping (≈ 1 s)
FOLLOW_SPEED = 150
FOLLOW_SETPOINT = 2000  # target value of the right sensor
FOLLOW_DIV = 10
SEEN_THRESH = 500       # above: right sensor sees the obstacle
LOST_THRESH = 100       # below (after having seen it): end of the side
RIGHT_SENSOR = 4
FRONT_SENSOR = 2

# === States ===
S_ADVANCE = 0       # approach the obstacle
S_TURN_L = 1        # first left turn
S_FOLLOW = 2        # wall following
S_PASS = 3          # drive past the edge
S_TURN_R = 4        # right turn at the corner
S_FINISH = 5        # short drive after the last turn
S_DONE = 6

# === Globals ===
state = 0
ticks = 0
error = 0
corr = 0
wall_seen = 0
turns = 0

timer_period[0] = 10


@onevent
def prox():
    global state, ticks, error, corr, wall_seen, turns
    global motor_left_target, motor_right_target

    if state == S_ADVANCE:
        if prox_horizontal[FRONT_SENSOR] > STOP_DIST:
            ticks = 0
            wall_seen = 0
            motor_left_target = -TURN_SPEED
            motor_right_target = TURN_SPEED
            state = S_TURN_L
        else:
            motor_left_target = SPEED
            motor_right_target = SPEED

    elif state == S_FOLLOW:
        if prox_horizontal[RIGHT_SENSOR] > SEEN_THRESH:
            wall_seen = 1

        if wall_seen == 1 and prox_horizontal[RIGHT_SENSOR] < LOST_THRESH:
            # End of the side: go past the edge
            ticks = 0
            motor_left_target = FOLLOW_SPEED
            motor_right_target = FOLLOW_SPEED
            state = S_PASS
        elif wall_seen == 1:
            error = prox_horizontal[RIGHT_SENSOR] - FOLLOW_SETPOINT
            corr = error // FOLLOW_DIV
            motor_left_target = FOLLOW_SPEED - corr
            motor_right_target = FOLLOW_SPEED + corr
        else:
            motor_left_target = FOLLOW_SPEED
            motor_right_target = FOLLOW_SPEED


@onevent
def timer0():
    global state, ticks, wall_seen, turns
    global motor_left_target, motor_right_target

    if state == S_TURN_L:
        ticks = ticks + 1
        if ticks >= TURN_TICKS:
            motor_left_target = FOLLOW_SPEED
            motor_right_target = FOLLOW_SPEED
            state = S_FOLLOW

    elif state == S_PASS:
        ticks = ticks + 1
        if ticks >= PASS_TICKS:
            ticks = 0
            motor_left_target = TURN_SPEED
            motor_right_target = -TURN_SPEED
            state = S_TURN_R

    elif state == S_TURN_R:
        ticks = ticks + 1
        if ticks >= TURN_TICKS:
            turns = turns + 1
            ticks = 0
            motor_left_target = FOLLOW_SPEED
            motor_right_target = FOLLOW_SPEED
            if turns >= NB_TURNS:
                state = S_FINISH
            else:
                wall_seen = 0
                state = S_FOLLOW

    elif state == S_FINISH:
        ticks = ticks + 1
        if ticks >= FINISH_TICKS:
            motor_left_target = 0
            motor_right_target = 0
            state = S_DONE

TranspilerError: unsupported constant 'advance' (line 12)

Exception in thread Thread-3:
Traceback (most recent call last):
  File "c:\Users\antoo\anaconda3\envs\thymio\Lib\threading.py", line 1045, in _bootstrap_inner
    self.run()
  File "c:\Users\antoo\anaconda3\envs\thymio\Lib\site-packages\tdmclient\tcp.py", line 73, in run
    packet = self.read_packet()
             ^^^^^^^^^^^^^^^^^^
  File "c:\Users\antoo\anaconda3\envs\thymio\Lib\site-packages\tdmclient\tcp.py", line 66, in read_packet
    raise error
  File "c:\Users\antoo\anaconda3\envs\thymio\Lib\site-packages\tdmclient\tcp.py", line 59, in read_packet
    packet_len = self.read_uint32()
                 ^^^^^^^^^^^^^^^^^^
  File "c:\Users\antoo\anaconda3\envs\thymio\Lib\site-packages\tdmclient\tcp.py", line 46, in read_uint32
    b = self.io.read(4)
        ^^^^^^^^^^^^^^^
  File "c:\Users\antoo\anaconda3\envs\thymio\Lib\site-packages\tdmclient\tcp.py", line 99, in read
    return self.socket.recv(n)
           ^^^^^^^^^^^^^^^^^^^
ConnectionResetError: [WinError 10054] Une conne